# News corpus (v2): building the news corpus

Revised version of `news_corpus.ipynb` (the first version, kept unchanged). What changed and why:

| Change | Reason |
|---|---|
| URL-slug pre-filter (AI word in the URL path) before sampling | Without it only ~9% of scraped articles were on-topic (45 of 900 URLs survived cleaning). In a test on 1,027 scraped articles, the slug filter raised the on-topic rate to ~33% |
| Region **Other** added | H3 names four regions: US, EU, UK, Other |
| Equal URLs per region (and per quarter if the window spans several) | Balanced design |
| Pilot decision uses the **on-topic** rate as well as the scrape-success rate | The old rule (40% scrapable) passed while 95% of the sample was off-topic |
| All paths relative to the repo root: URL list and log in `dataset/news/`, article text in `data/` (git-ignored) | Old notebook wrote to `data/` but the committed files were in `dataset/news/` |
| No `!pip install`: use `pip install -r requirements.txt` | One dependency list for everyone |
| Contact address read from `.env` (`NEWS_CONTACT`) | No personal email in the public repo |
| `scraped.csv` rows not in the current sample are reported and ignored | An old run had left 889 stray rows |

**Known limitation (write it up):** the slug filter selects articles that name AI in the URL. Those tend to be AI-focused pieces, so frames in the corpus may differ from articles where AI is a side topic. It is applied identically to every region.

Steps: 1 load URL list, 2 filter and sample, 3 scrape functions, 4 pilot, 5 full scrape, 6 clean, 7 review rejects, 8 log.

In [1]:
import json, re, os, time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
import trafilatura
import py3langid as langid
from dotenv import load_dotenv

load_dotenv(Path(".env"))           # reads .env from the folder the notebook runs in (the repo root)
Path("data").mkdir(exist_ok=True)    # article text lives here: git-ignored, never committed
Path("dataset/news").mkdir(parents=True, exist_ok=True)   # URL list + log: committed

## Settings
Change these if needed and write the values in the report.
Set the months only after checking the project plan: the dataset covers 2022-11 to 2026-06.

In [2]:
START_MONTH = "2025-07"      # first month to keep (EU and UK are small: Apr-Jun 2026 alone gives only ~260 / ~290 URLs)
END_MONTH   = "2026-06"      # last month to keep
REGIONS     = ["US", "UK", "EU", "Other"]
N_PER_REGION = 600           # URLs to scrape per region (split evenly across quarters)
N_PILOT     = 60             # pilot URLs (spread over regions)
MAX_PER_OUTLET = 15          # so a few big sites don't dominate
MIN_WORDS, MAX_WORDS = 150, 3000
SEED        = 42

# Pilot decision: both must hold
PILOT_MIN_USABLE  = 0.40     # share of pilot URLs that give usable English text
PILOT_MIN_ONTOPIC = 0.10     # share of pilot URLs that are also on-topic

NEWS_CONTACT = os.environ.get("NEWS_CONTACT", "")
if not NEWS_CONTACT:
    raise SystemExit("Add NEWS_CONTACT=your-university-email to .env (used in the scraper User-Agent).")
USER_AGENT = f"NCI-student-research/1.0 (contact: {NEWS_CONTACT})"

# Pre-filter on the URL path (before scraping). Matched against lower-cased tokens of the slug.
AI_SLUG = re.compile(r"\b(?:ai|artificial intelligence|chatbots?|chatgpt|openai|llms?|gemini|claude|"
                     r"anthropic|deepfakes?|generative|copilot)\b")

# Domain endings kept for regions where most outlets are not English. Language is checked again after scraping.
ENGLISH_TLDS = {
    "EU":    {"ie", "eu", "mt", "com", "net", "org", "news"},
    "Other": {"com", "net", "org", "news", "in", "au", "ca", "nz", "za", "ng", "ke", "sg", "ph", "pk", "my", "gh", "io"},
}

## 1. Load the URL list
About 90 MB. It downloads once and is then read from disk.

In [3]:
CFG = json.load(open("config.json"))                    # all data paths live in config.json
PARQUET = Path(CFG["parquet"])
URL = "https://huggingface.co/datasets/brewcoua/genai-gdelt-framing/resolve/main/articles.parquet"

if not PARQUET.exists():
    r = requests.get(URL, timeout=300)
    r.raise_for_status()
    PARQUET.write_bytes(r.content)

df = pd.read_parquet(PARQUET, columns=["document_id", "month", "region"])
df = df.rename(columns={"document_id": "url"})
print(len(df), "articles in the dataset")
print(df["region"].value_counts().to_dict())

1116091 articles in the dataset
{'Other': 526661, 'US': 296721, 'EU': 250099, 'UK': 42610}


## 2. Filter and sample
Each step is counted in `steps` so the numbers go straight into the report.

In [4]:
steps = [("all articles", len(df))]

df["month"] = df["month"].astype(str)
df = df[(df["month"] >= START_MONTH) & (df["month"] <= END_MONTH)]
steps.append((f"months {START_MONTH} to {END_MONTH}", len(df)))

df = df[df["region"].isin(REGIONS)]
steps.append(("regions " + ", ".join(REGIONS), len(df)))

# Remove tracking bits (?utm=...) so the same article isn't counted twice
df["url"] = df["url"].str.split("?").str[0].str.rstrip("/")
df = df[df["url"].str.startswith("http")].drop_duplicates("url")
steps.append(("unique URLs", len(df)))

# --- topic pre-filter on the URL path (see the note at the top about its limitation) ---
slug = df["url"].str.replace(r"^https?://[^/]+", "", regex=True).str.lower()
slug_tokens = slug.str.replace(r"[^a-z0-9]+", " ", regex=True)
df = df[slug_tokens.str.contains(AI_SLUG.pattern)]
steps.append(("AI word in URL path", len(df)))

df["outlet"] = df["url"].map(lambda u: urlsplit(u).netloc.lower().removeprefix("www."))
tld = df["outlet"].str.split(".").str[-1]
for region, allowed in ENGLISH_TLDS.items():
    df = df[(df["region"] != region) | tld.loc[df.index].isin(allowed)]
steps.append(("EU/Other: likely-English domains", len(df)))

df = df.sample(frac=1, random_state=SEED)
df = df[df.groupby("outlet").cumcount() < MAX_PER_OUTLET]
steps.append((f"max {MAX_PER_OUTLET} per outlet", len(df)))

# Equal number of URLs per region, split evenly across quarters
df["quarter"] = pd.PeriodIndex(df["month"], freq="M").asfreq("Q").astype(str)
n_quarters = df["quarter"].nunique()
per_cell = N_PER_REGION // n_quarters
sample = df.groupby(["region", "quarter"]).head(per_cell)
steps.append(("full sample", len(sample)))

short = sample.groupby("region").size()
if (short < per_cell * n_quarters).any():
    print("WARNING: some region/quarter cells have fewer URLs than requested:\n", short)

pilot = sample.groupby("region").head(N_PILOT // len(REGIONS))

sample.to_csv(CFG["urls_full_csv"], index=False)
pilot.to_csv(CFG["urls_pilot_csv"], index=False)

display(pd.DataFrame(steps, columns=["step", "rows"]))
pd.crosstab(sample["quarter"], sample["region"], margins=True)

 region
EU       600
Other    600
UK       560
US       600
dtype: int64


,step,rows
0,all articles,1116091
1,months 2025-07 to 2026-06,314876
2,"regions US, UK, EU, Other",314876
3,unique URLs,309066
4,AI word in URL path,37832
5,EU/Other: likely-English domains,33206
6,max 15 per outlet,17952
7,full sample,2360


region,EU,Other,UK,US,All
quarter,,,,,
2025Q3,150,150,110,150,560
2025Q4,150,150,150,150,600
2026Q1,150,150,150,150,600
2026Q2,150,150,150,150,600
All,600,600,560,600,2360


## 3. Scraping functions
Results are saved to `data/scraped.csv` every 25 URLs. URLs already in that file are skipped.

In [5]:
HEADERS = {"User-Agent": USER_AGENT}
SCRAPED = Path(CFG["scraped_csv"])
PAYWALL = re.compile(r"subscribe (now|to continue|to read)|already a subscriber|sign in to continue", re.I)
robots_cache = {}

def robots_ok(url):
    site = "https://" + urlsplit(url).netloc
    if site not in robots_cache:
        rp = RobotFileParser()
        try:
            r = requests.get(site + "/robots.txt", headers=HEADERS, timeout=10)
            rp.parse(r.text.splitlines() if r.status_code == 200 else [])
        except requests.RequestException:
            rp.parse([])
        robots_cache[site] = rp
    return robots_cache[site].can_fetch(USER_AGENT, url)

def scrape(url):
    result = {"url": url, "status": "", "title": "", "date": "", "text": "",
              "n_words": 0, "lang": "", "accessed": datetime.now(timezone.utc).isoformat()}
    if not robots_ok(url):
        result["status"] = "robots_blocked"
        return result
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
    except requests.RequestException:
        result["status"] = "failed"
        return result
    if r.status_code != 200:
        result["status"] = f"http_{r.status_code}"
        return result

    out = trafilatura.extract(r.text, output_format="json", with_metadata=True,
                              include_comments=False, include_tables=False)
    if not out:
        result["status"] = "no_text"
        return result
    out = json.loads(out)
    text = out.get("text") or ""
    result.update(title=out.get("title") or "", date=out.get("date") or "",
                  text=text, n_words=len(text.split()), lang=langid.classify(text)[0])

    if result["n_words"] < MIN_WORDS:
        result["status"] = "too_short"
    elif result["n_words"] < 400 and PAYWALL.search(text):
        result["status"] = "paywall"
    elif result["lang"] != "en":
        result["status"] = "not_english"
    else:
        result["status"] = "ok"
    return result

def scrape_all(urls):
    urls = list(urls)
    done = pd.read_csv(SCRAPED, keep_default_na=False) if SCRAPED.exists() else pd.DataFrame(columns=["url"])
    stray = ~done["url"].isin(set(sample["url"]))
    if stray.any():
        print(f"NOTE: {stray.sum()} rows in scraped.csv are not in the current sample (old run?). They are ignored.")
    todo = [u for u in urls if u not in set(done["url"])]
    print(len(urls) - len(todo), "already done,", len(todo), "to go")
    new = []
    for i, url in enumerate(todo, 1):
        new.append(scrape(url))
        time.sleep(1)                                   # be polite
        if i % 25 == 0 or i == len(todo):
            done = pd.concat([done, pd.DataFrame(new)], ignore_index=True)
            done.to_csv(SCRAPED, index=False)
            new = []
            print(f"{i}/{len(todo)}")
    return done[done["url"].isin(urls)]

## Topic rule (used by the pilot and by cleaning)
At least 2 AI words and at least 2 regulation words in the article text. The list is separate from the frame keywords, so the filter does not decide which frames appear.
The regulation list is loose (`ban`, `bills`, `rules`, `laws` also match everyday news). It is kept as in v1 until section 7 has been read; tighten it only with evidence.

In [6]:
AI_WORDS  = re.compile(r"\b(AI|artificial intelligence|chatbots?|ChatGPT|large language models?|LLMs?|OpenAI|Gemini)\b")
REG_WORDS = re.compile(r"\b(regulat\w*|legislat\w*|laws?|AI Act|bills?|rules?|oversight|ban|governance|policy|lawmakers?)\b", re.I)

def topic_counts(frame):
    frame = frame.copy()
    frame["ai_hits"]  = frame["text"].map(lambda t: len(AI_WORDS.findall(t)))
    frame["reg_hits"] = frame["text"].map(lambda t: len(REG_WORDS.findall(t)))
    frame["on_topic"] = (frame["ai_hits"] >= 2) & (frame["reg_hits"] >= 2)
    return frame

## 4. Pilot, then the decision
Scrapes ~60 URLs. Run this before the full scrape.

In [7]:
res = scrape_all(pilot["url"])
res = topic_counts(res.merge(pilot[["url", "region"]], on="url"))
res["usable"] = (res["status"] == "ok") & (res["n_words"] <= MAX_WORDS)

print(res["status"].value_counts(), "\n")
print(res.groupby("region")[["usable", "on_topic"]].mean().round(2))

usable_rate  = res["usable"].mean()
ontopic_rate = (res["usable"] & res["on_topic"]).mean()
print(f"\nUsable: {usable_rate:.0%} (need {PILOT_MIN_USABLE:.0%})   On-topic: {ontopic_rate:.0%} (need {PILOT_MIN_ONTOPIC:.0%})")
go = usable_rate >= PILOT_MIN_USABLE and ontopic_rate >= PILOT_MIN_ONTOPIC
print("Decision:", "scale up to the full list" if go else "below threshold: revisit the slug filter / months before scraping")

60 already done, 0 to go
status
ok                40
not_english        8
http_403           8
http_451           1
failed             1
robots_blocked     1
too_short          1
Name: count, dtype: int64 

        usable  on_topic
region                  
EU         0.6      0.33
Other      0.8      0.27
UK         0.6      0.07
US         0.6      0.33

Usable: 65% (need 40%)   On-topic: 22% (need 10%)
Decision: scale up to the full list


## 5. Full scrape
About 2,400 URLs (4 regions x 600) at one request per second: roughly 1 hour. Safe to stop and rerun; finished URLs are skipped.

In [8]:
res = scrape_all(sample["url"])
print(res["status"].value_counts())

1510 already done, 850 to go


25/850


50/850


75/850


100/850


...

750/850


775/850


800/850


825/850


850/850
status
ok                1564
not_english        274
http_403           250
http_404            61
http_429            39
robots_blocked      38
too_short           35
failed              33
http_451            21
http_410            16
no_text             13
http_406             7
http_500             3
paywall              3
http_503             1
http_307             1
http_525             1
Name: count, dtype: int64


## 6. Clean and save
- keep only usable texts (`ok`), 150–3,000 words (very long pages are usually live blogs)
- on-topic rule above
- drop duplicates (same text, or same title from the same wire story)

`news_clean.csv` goes to `data/` (git-ignored: the article text belongs to the publishers).

In [9]:
news = res.merge(sample[["url", "outlet", "region", "month", "quarter"]], on="url")
clean_steps = [("scraped", len(news))]

news = news[news["status"] == "ok"]
clean_steps.append(("usable text", len(news)))

news = news[news["n_words"] <= MAX_WORDS]
clean_steps.append((f"<= {MAX_WORDS} words", len(news)))

news = topic_counts(news)
rejected = news[~news["on_topic"]]          # kept for section 7
news = news[news["on_topic"]]
clean_steps.append(("on-topic", len(news)))

news["text_key"] = news["text"].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
news = news.drop_duplicates("text_key")
news = news[(news["title"] == "") | ~news.duplicated("title")]
clean_steps.append(("no duplicates", len(news)))

news["date"] = news["date"].where(news["date"] != "", news["month"])
news["n_words"] = news["n_words"].astype(int)
news = news.reset_index(drop=True)
news["doc_id"] = [f"news_{i:04d}" for i in range(1, len(news) + 1)]
news = news[["doc_id", "text", "outlet", "title", "date", "month", "quarter", "region", "url", "n_words", "accessed"]]
news.to_csv(CFG["news_corpus_csv"], index=False)

display(pd.DataFrame(clean_steps, columns=["step", "rows"]))
print(news["outlet"].nunique(), "outlets, median", int(news["n_words"].median()), "words")
pd.crosstab(news["quarter"], news["region"], margins=True)

,step,rows
0,scraped,2360
1,usable text,1564
2,<= 3000 words,1535
3,on-topic,504
4,no duplicates,420


285 outlets, median 928 words


region,EU,Other,UK,US,All
quarter,,,,,
2025Q3,13,16,16,32,77
2025Q4,21,23,28,31,103
2026Q1,22,35,23,36,116
2026Q2,26,30,34,34,124
All,82,104,101,133,420


## 7. Review the rejects
Two questions to answer by reading, before deciding whether to loosen or tighten the topic rule:
1. Articles with plenty of AI words but fewer than 2 regulation words: AI news, or regulation framing the regex missed?
2. Articles that pass: do they really discuss AI regulation?

Write down how many of 30 you judged on-topic. That number goes in the report.

In [10]:
ai_only = rejected[(rejected["ai_hits"] >= 2) & (rejected["reg_hits"] < 2)]
print(len(ai_only), "articles have 2+ AI words but fewer than 2 regulation words")
ai_only.sample(min(30, len(ai_only)), random_state=SEED)[["region", "outlet", "title", "ai_hits", "reg_hits", "url"]]

988 articles have 2+ AI words but fewer than 2 regulation words


,region,outlet,title,ai_hits,reg_hits,url
477,Other,businessdayghana.com,Alibaba Unveils Cutting-Edge AI Coding Model Q...,5,0,https://businessdayghana.com/alibaba-unveils-c...
738,EU,kyuk.org,Tech CEOs say the era of 'code by AI' is here....,47,0,https://www.kyuk.org/2025-10-21/tech-ceos-say-...
1153,US,thewrap.com,AI Surprise in Cannes: Curiosity Wins Over Fear,29,0,https://www.thewrap.com/creative-content/movie...
692,Other,ktex.iheart.com,Nearly 3 In 10 Teens Say They Use AI Chatbots ...,26,0,https://ktex.iheart.com/content/2025-12-09-nea...
211,Other,philippinetimes.com,A New Chapter in Taiwan-Japan Tech Collaborati...,7,0,http://www.philippinetimes.com/news/278959075/...
1235,US,electroiq.com,"Claude Vs. ChatGPT Statistics By User, Website...",94,0,https://electroiq.com/stats/claude-vs-chatgpt-...
628,Other,zdnet.com,Is the new Microsoft 365 Premium worth it? Her...,16,0,https://www.zdnet.com/article/is-the-new-micro...
250,US,miragenews.com,Berkeley Lab Leads AI Effort for Energy Materials,42,0,https://www.miragenews.com/berkeley-lab-leads-...
136,Other,techcrunch.com,Notion just turned its workspace into a hub fo...,11,0,https://techcrunch.com/2026/05/13/notion-just-...
1349,EU,technology.inquirer.net,When AI blurs magic and technology,12,0,https://technology.inquirer.net/146991/when-ai...


In [11]:
passed = topic_counts(news)
passed.sample(min(30, len(passed)), random_state=SEED)[["region", "outlet", "title", "ai_hits", "reg_hits", "url"]]

,region,outlet,title,ai_hits,reg_hits,url
145,Other,oklahomastar.com,Red Hat Launches New Developer Tools for Agent...,21,3,http://www.oklahomastar.com/news/279077656/red...
334,EU,singaporestar.com,"""Trust Is the New Infrastructure"": A Featured ...",2,5,http://www.singaporestar.com/news/278845967/tr...
175,US,telecom.economictimes.indiatimes.com,Pentagon casts cloud of doubt over Anthropic’s...,17,5,https://telecom.economictimes.indiatimes.com/n...
369,EU,dazeddigital.com,"You, me and ChatGPT: How AI-speak became a twi...",54,5,https://www.dazeddigital.com/life-culture/arti...
416,UK,punchng.com,What Nigerian National AI Act could look like,45,25,https://punchng.com/what-nigerian-national-ai-...
73,Other,manilatimes.net,"Zetrix, CAICT's Astron Unveil Blockchain-AI Tr...",32,2,https://www.manilatimes.net/2026/04/15/tmt-new...
132,US,mercurynews.com,"Larry Magid: AI isn’t human, but it can feel a...",29,2,https://www.mercurynews.com/2026/04/23/larry-m...
137,Other,globalnews.ca,AI tokens are catching some companies by surpr...,29,2,https://globalnews.ca/news/11896609/artificial...
30,US,abc13.com,Trump signs an executive order to vet top AI m...,10,3,https://abc13.com/post/trump-signs-executive-o...
72,Other,fonearena.com,Anthropic rolls out Claude Fable 5 and Claude ...,2,2,https://www.fonearena.com/blog/484689/anthropi...


## 8. Log for the report
Saves the settings, filter counts and access dates for the data section.

In [12]:
log = {
    "dataset": "brewcoua/genai-gdelt-framing (CC BY 4.0)",
    "settings": {"months": [START_MONTH, END_MONTH], "regions": REGIONS, "n_per_region": N_PER_REGION,
                 "max_per_outlet": MAX_PER_OUTLET, "min_words": MIN_WORDS, "max_words": MAX_WORDS, "seed": SEED,
                 "ai_slug_filter": AI_SLUG.pattern},
    "selection_steps": steps,
    "cleaning_steps": clean_steps,
    "pilot": {"usable": float(usable_rate), "on_topic": float(ontopic_rate)},
    "access_dates": [news["accessed"].min(), news["accessed"].max()],
    "on_topic_rule": ">= 2 AI words and >= 2 regulation words",
    "limitation": "URL-slug pre-filter selects articles that name AI in the URL.",
}
Path(CFG["corpus_log_json"]).write_text(json.dumps(log, indent=2, default=str))
log

{'dataset': 'brewcoua/genai-gdelt-framing (CC BY 4.0)',
 'settings': {'months': ['2025-07', '2026-06'],
  'regions': ['US', 'UK', 'EU', 'Other'],
  'n_per_region': 600,
  'max_per_outlet': 15,
  'min_words': 150,
  'max_words': 3000,
  'seed': 42,
  'ai_slug_filter': '\\b(?:ai|artificial intelligence|chatbots?|chatgpt|openai|llms?|gemini|claude|anthropic|deepfakes?|generative|copilot)\\b'},
 'selection_steps': [('all articles', 1116091),
  ('months 2025-07 to 2026-06', 314876),
  ('regions US, UK, EU, Other', 314876),
  ('unique URLs', 309066),
  ('AI word in URL path', 37832),
  ('EU/Other: likely-English domains', 33206),
  ('max 15 per outlet', 17952),
  ('full sample', 2360)],
 'cleaning_steps': [('scraped', 2360),
  ('usable text', 1564),
  ('<= 3000 words', 1535),
  ('on-topic', 504),
  ('no duplicates', 420)],
 'pilot': {'usable': 0.65, 'on_topic': 0.21666666666666667},
 'access_dates': ['2026-10-26T21:23:39.082354+00:00',
  '2026-10-27T10:36:28.170625+00:00'],
 'on_topic_rule':

**Licensing note:** the article text belongs to the publishers. Keep `data/scraped.csv` and `data/news_clean.csv` out of the repo and out of the submission ZIP. Share `dataset/news/urls_full.csv`, this notebook and `dataset/news/corpus_log.json` so others can rebuild the corpus.